# Model Serving with FastAPI & Docker

## Objective

In Notebook 06, I recorded the fitted treatment and control models in MLflow and verified that the saved artifacts reproduced their original predictions.

In this notebook, I turn those models into a reusable API.

The API accepts customer features, returns predicted treatment and control probabilities, calculates uplift, and applies the budget-constrained contact policy developed in Notebook 05.

I use a versioned serving bundle exported from the MLflow model references. This separates inference from the local MLflow tracking database and avoids retraining the model during API startup.

I will verify that API predictions match the original scoring results, test the service's input validation and decision logic, and run the same application inside Docker.

The service is a local development demonstration, not a publicly deployed production system.

## 1. Serving architecture

The T-learner contains two independently fitted outcome models.

For an incoming customer, the API calculates:

\[
\widehat{p}_1(X)
=
\widehat{P}(Y=1\mid X,T=1)
\]

\[
\widehat{p}_0(X)
=
\widehat{P}(Y=1\mid X,T=0)
\]

The predicted uplift is:

\[
\widehat{\tau}(X)
=
\widehat{p}_1(X)-\widehat{p}_0(X)
\]

The API then passes the resulting scores to the same decision engine used in Notebook 05.

This separation matters because the model estimates customer response while the decision engine applies business assumptions and budget constraints. Changing the contact budget or conversion value does not require retraining the model.

In [1]:
# ============================================================
# Verify the portable serving artifact
#
# The export script must have completed successfully.
# This cell verifies that the deployment files exist and
# identifies the specific MLflow run being served.
# ============================================================

from pathlib import Path
import json

PROJECT_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "dbt" / "dbt_project.yml").exists()
)

DEPLOYMENT_DIR = (
    PROJECT_ROOT
    / "data"
    / "deployment"
)

BUNDLE_PATH = (
    DEPLOYMENT_DIR
    / "model_bundle.joblib"
)

MANIFEST_PATH = (
    DEPLOYMENT_DIR
    / "deployment_manifest.json"
)

SAMPLE_PATH = (
    DEPLOYMENT_DIR
    / "sample_request.json"
)

assert BUNDLE_PATH.exists()
assert MANIFEST_PATH.exists()
assert SAMPLE_PATH.exists()

manifest = json.loads(
    MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

print(
    "MLflow run:",
    manifest["mlflow_run_id"],
)

print(
    "Feature count:",
    manifest["feature_count"],
)

print("Deployment files verified.")

MLflow run: 11a6f74e47e240b8a76c4aa0646a1419
Feature count: 34
Deployment files verified.


## 2. API acceptance testing

I test the service using the same customer features previously scored in Notebook 05.

First, I confirm that the API reports the expected MLflow run and feature count. Then I submit a customer to the scoring endpoint and compare the returned probabilities against the saved scoring cache.

This verifies that the API is serving the intended model version and applying the same feature order and prediction logic.

I also test the decision endpoint using hypothetical conversion value, contact cost, and budget inputs.

These checks verify inference consistency and application behavior. They do not constitute an independent evaluation of uplift accuracy or causal effectiveness.

In [2]:
# ============================================================
# API acceptance test against Notebook 05 predictions
#
# The API must already be running on localhost:8000.
# ============================================================

import numpy as np
import pandas as pd
import httpx


BASE_URL = "http://127.0.0.1:8000"

sample_request = json.loads(
    SAMPLE_PATH.read_text(
        encoding="utf-8"
    )
)


# ------------------------------------------------------------
# Confirm the service and model version
# ------------------------------------------------------------

with httpx.Client(
    base_url=BASE_URL,
    timeout=30.0,
) as client:

    health_response = client.get(
        "/health"
    )

    health_response.raise_for_status()

    model_response = client.get(
        "/model-info"
    )

    model_response.raise_for_status()

    score_response = client.post(
        "/score",
        json=sample_request,
    )

    score_response.raise_for_status()


health = health_response.json()
model_info = model_response.json()
api_result = score_response.json()


assert health["status"] == "ok"

assert (
    model_info["mlflow_run_id"]
    == manifest["mlflow_run_id"]
)

assert (
    model_info["feature_count"]
    == manifest["feature_count"]
)


# ------------------------------------------------------------
# Compare the API result against the prediction cache
# from Notebook 05.
# ------------------------------------------------------------

prediction_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "uplift_scoring_predictions.parquet"
)

saved_predictions = pd.read_parquet(
    prediction_path
)

saved_predictions.columns = (
    saved_predictions.columns.str.lower()
)

api_prediction = api_result[
    "predictions"
][0]

expected_prediction = (
    saved_predictions.loc[
        saved_predictions["client_id"]
        == api_prediction["client_id"]
    ].iloc[0]
)

for column in [
    "p_treatment",
    "p_control",
    "predicted_uplift",
]:

    np.testing.assert_allclose(
        api_prediction[column],
        expected_prediction[column],
        rtol=1e-10,
        atol=1e-10,
    )


print("PASS: API is healthy.")
print("PASS: MLflow model version matches.")
print("PASS: API predictions match Notebook 05.")

display(
    pd.DataFrame(
        api_result["predictions"]
    )
)

PASS: API is healthy.
PASS: MLflow model version matches.
PASS: API predictions match Notebook 05.


,client_id,p_treatment,p_control,predicted_uplift
0,807454185f,0.492883,0.469827,0.023056


In [3]:
# ============================================================
# Decision endpoint acceptance test
#
# Reuse the sample customer and add hypothetical economics.
# This checks the API's integration with src/decisioning.py.
# ============================================================

decision_request = {
    **sample_request,
    "conversion_value": 20.0,
    "contact_cost": 0.25,
    "budget": 10.0,
}

with httpx.Client(
    base_url=BASE_URL,
    timeout=30.0,
) as client:

    response = client.post(
        "/decide",
        json=decision_request,
    )

    response.raise_for_status()

decision_result = response.json()


assert (
    decision_result["summary"]["contact_spend"]
    <= decision_request["budget"]
)

assert (
    decision_result["mlflow_run_id"]
    == manifest["mlflow_run_id"]
)


print("PASS: Decision request completed.")
print("PASS: Contact budget respected.")

display(
    pd.DataFrame(
        decision_result["decisions"]
    )
)

PASS: Decision request completed.
PASS: Contact budget respected.


,client_id,predicted_uplift,p_treatment,modeled_incremental_value,modeled_net_value_per_contact,contact
0,807454185f,0.023056,0.492883,0.461119,0.211119,True


## 3. Containerized inference

The API works in my local Python environment, but that environment also contains development tools such as JupyterLab, dbt, Snowflake libraries, and MLflow.

Docker allows me to package the serving application with a smaller, explicitly defined set of dependencies.

The container receives a pre-exported, versioned model bundle. It does not require access to Snowflake, the training dataset, or the local MLflow tracking database.

For this local demonstration, I mount the deployment artifacts into the container as read-only files. This keeps the model artifacts outside Git and outside the Docker image.

The container still depends on compatible Python and scikit-learn versions, so I use the pinned serving requirements and verify prediction consistency after starting it.

In [ ]:
# ============================================================
# Final container acceptance check
#
# This request should succeed against the Docker-hosted API
# exactly as it did against the local Uvicorn process.
# ============================================================

with httpx.Client(
    base_url="http://127.0.0.1:8000",
    timeout=30.0,
) as client:

    health_response = client.get(
        "/health"
    )

    score_response = client.post(
        "/score",
        json=sample_request,
    )

health_response.raise_for_status()
score_response.raise_for_status()

container_prediction = (
    score_response.json()["predictions"][0]
)


for column in [
    "p_treatment",
    "p_control",
    "predicted_uplift",
]:

    np.testing.assert_allclose(
        container_prediction[column],
        expected_prediction[column],
        rtol=1e-10,
        atol=1e-10,
    )


print("PASS: Docker API is healthy.")
print("PASS: Container predictions match Notebook 05.")
print("PASS: Serving model version is unchanged.")

## Conclusions and limitations

I converted the versioned logistic T-learner from Notebook 06 into a reusable inference service.

I exported the exact treatment and control models recorded in MLflow, preserved their model references and feature schema, and packaged them into a portable serving bundle.

The FastAPI application exposes four endpoints: health, model information, customer uplift scoring, and budget-constrained treatment decisioning. Both prediction endpoints use the same underlying scoring function, and the decision endpoint reuses the existing decision engine from Notebook 05.

I verified that the API reproduced the treatment, control, and uplift predictions saved in Notebook 05. I also added API tests covering model loading, prediction results, missing-feature validation, and budget constraints.

Finally, I ran the application in Docker using pinned serving dependencies and a read-only model-artifact mount.

### Limitations

The API is currently a local development service. It does not provide authentication, rate limiting, multi-user access controls, or production monitoring.

The serving bundle is exported from a local MLflow tracking environment rather than a shared production model registry. The deployment process is not yet automated through CI/CD.

The API returns model predictions, not independently verified individual causal effects. Its decision outputs also depend on hypothetical conversion values, contact costs, and budgets.

The next stage is to automate data and application workflows, introduce reliability controls, and add monitoring for model and data quality.